# Topic Modeling with BERTopic — SynBio Papers

Loads the pre-computed **SynBio Papers** embeddings, fits a BERTopic model
(UMAP → HDBSCAN) with manually-set hyperparameters, inspects the result, and
saves the fitted model, the topic-info table, and document-level topic
assignments to `assets/topic_models/`.

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import numpy as np

from aux.paths import MODELS_DIR, set_seed
from aux.topic_modeling import load_corpus, fit_topic_model, save_topic_outputs

set_seed()

# ── CONFIG: SynBio Papers ──────────────────────────────────────────────────
EMBEDDINGS_FILE  = "papers_embeddings.npy"
CORPUS_FILE      = "papers_corpus.txt"
ID_COL           = "id"
PREFIX           = "papers"
MIN_CLUSTER_SIZE = 30                 # HDBSCAN minimum cluster size

## 1. Load embeddings and corpus

In [ ]:
embeddings, corpus = load_corpus(EMBEDDINGS_FILE, CORPUS_FILE)
docs = corpus["text"].tolist()
print(f"Papers: {embeddings.shape[0]:,} docs, {embeddings.shape[1]} dims")

## 2. Fit BERTopic

In [ ]:
model, topics, probs = fit_topic_model(
    docs, embeddings, min_cluster_size=MIN_CLUSTER_SIZE, verbose=True,
)

## 3. Inspect

In [ ]:
info = model.get_topic_info()
n_topics = info.Topic.max() + 1
print(f"Papers — topics found: {n_topics}  |  outliers: {(np.array(topics) == -1).sum():,}")
info.head(20)

## 4. Save model and summaries

In [ ]:
save_topic_outputs(model, corpus, topics, ID_COL, PREFIX)
print(f"Saved → {MODELS_DIR}")
for f in sorted(MODELS_DIR.glob(f"{PREFIX}_*")):
    print(f"  {f.name}")